<a href="https://colab.research.google.com/github/diegosolgtz-pixel/-Regresan-los-usuarios-de-RappiPlus-/blob/main/Analisis_Rappiplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida


- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog= pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')


In [3]:
# explorar datasets
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
orders.info()       # tipos de datos y valores nulos
orders.isnull().sum()  # conteo de nulos por columna
orders.duplicated().sum()  # conteo de duplicados

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


np.int64(100)

In [5]:
orders[['cantidad','precio_unitario','monto_descuento','monto_total']].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [6]:
# Investigar el negativo y el outlier de 20000
anomalias = orders[(orders['cantidad'] <= 0) | (orders['cantidad'] == 20000)]
print(anomalias[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']])

      cantidad  precio_unitario  monto_descuento  monto_total
266       -2.0           101.31             10.0      -192.62
267       -1.0            43.50              5.0       -38.50
268       -1.0           497.65              5.0      -492.65
269       -1.0           423.53              0.0      -423.53
3656   20000.0           297.66              0.0   5953200.00
3668   20000.0           348.31              0.0   6966200.00
3722   20000.0           442.01              0.0   8840200.00
3726   20000.0           290.85              0.0   5817000.00


In [7]:
orders.isnull().sum()

,0
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,300
dispositivo,20
fuente_referencia,30
nombre_producto,30
categoria_producto,80
cantidad,50
precio_unitario,50


In [8]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [9]:
catalog.info()       # tipos de datos y valores nulos
catalog.isnull().sum()  # conteo de nulos por columna
catalog.duplicated().sum()  # conteo de duplicados

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 356.0+ bytes


np.int64(0)

In [10]:
catalog['costo_unitario'].describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [11]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [12]:
marketing.info()       # tipos de datos y valores nulos
marketing.isnull().sum()  # conteo de nulos por columna
marketing.duplicated().sum()  # conteo de duplicados

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


np.int64(0)

In [13]:
marketing.isnull().sum()

,0
fecha,0
pais,0
id_campaña,0
canal,101
gasto,0


---

### Revisión y calidad de datos

 Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [14]:
orders['fecha_hora_pedido']= pd.to_datetime(orders['fecha_hora_pedido'], errors="coerce")
marketing['fecha']= pd.to_datetime(marketing['fecha'], errors="coerce")

In [15]:
# Verificar si los 50 datos ausentes en cantidad, precio unitario y monto descuento son de las mimsmas filas
# Filtrar las filas donde las tres columnas clave son nulas al mismo tiempo
filas_rotas = orders[
    orders['cantidad'].isna() &
    orders['precio_unitario'].isna() &
    orders['monto_descuento'].isna()
]

# Comprobar cuántas filas cumplen con esta condición exacta
print(f"Número de filas donde las 3 columnas son nulas a la vez: {len(filas_rotas)}")

# Echar un vistazo a las primeras filas para confirmarlo visualmente
filas_rotas.head()

Número de filas donde las 3 columnas son nulas a la vez: 50


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39


In [16]:
# 1. Eliminar los 50 nulos usando .dropna() solo en la columna 'cantidad'
orders = orders.dropna(subset=['cantidad'])

# 2. Eliminar los datos negativos y las 20000 unidades con un filtro
orders = orders[(orders['cantidad'] > 0) & (orders['cantidad'] != 20000)]

# 3. Comprobamos cómo quedó el contador de nulos en esas columnas
print("--- Conteo de nulos actualizado ---")
print(orders[['cantidad', 'precio_unitario', 'monto_descuento']].isnull().sum())

print("\n--- Resumen estadístico de cantidad ---")
print(orders['cantidad'].describe())

--- Conteo de nulos actualizado ---
cantidad           0
precio_unitario    0
monto_descuento    0
dtype: int64

--- Resumen estadístico de cantidad ---
count    25042.000000
mean         3.900567
std        154.751426
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      10000.000000
Name: cantidad, dtype: float64


In [17]:
valores_unicos = orders['cantidad'].unique()
print(valores_unicos)

[2.e+00 1.e+00 1.e+04]


In [18]:
# Eliminar cantidaes de 10000
orders = orders[orders['cantidad'] < 10000]

# Comprobación final
print("Valores únicos finales:")
print(orders['cantidad'].unique())

print("\nResumen estadístico final:")
print(orders['cantidad'].describe())

Valores únicos finales:
[2. 1.]

Resumen estadístico final:
count    25036.000000
mean         1.504953
std          0.499985
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          2.000000
Name: cantidad, dtype: float64


In [19]:
# En cuanto al resto de nulos:
#Eliminamos únicamente las filas donde falta el producto o la categoría
orders = orders.dropna(subset=['nombre_producto', 'categoria_producto'])

# 2. Para las demás columnas que no se requieren para calculos posteriores, rellenamos con un texto genérico
# Así conservamoa el 'monto_total' intacto para sumas de ingresos
orders['pais'] = orders['pais'].fillna('No Registrado')
orders['dispositivo'] = orders['dispositivo'].fillna('No Registrado')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('No Registrado')

# 3. Comprobación final
print("--- Conteo final de nulos en orders ---")
print(orders.isnull().sum())

--- Conteo final de nulos en orders ---
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64


In [20]:
# Ver si hay precios en cero o negativos
precios_raros = orders[orders['precio_unitario'] <= 0]
print(f"Filas con precios inválidos: {len(precios_raros)}")

# Ver si hay algún descuento que supere el monto total de manera ilógica
descuentos_raros = orders[orders['monto_descuento'] > (orders['cantidad'] * orders['precio_unitario'])]
print(f"Filas con descuentos ilógicos: {len(descuentos_raros)}")

Filas con precios inválidos: 0
Filas con descuentos ilógicos: 0


In [21]:
# Contar cuántos duplicados totales hay antes de borrar
print("Duplicados en orders:", orders.duplicated().sum())

Duplicados en orders: 100


In [22]:
#borrar los 100 duplicados conservando la primera aparición
orders = orders.drop_duplicates()

In [23]:
# Quitar espacios en blanco invisibles al inicio o final de los textos
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip()

# Ver los productos únicos para asegurar que no haya repetidos con nombres distintos
print(catalog['nombre_producto'].unique())

['Laptop-Gaming-16GB' 'Phone-Pro-128GB' 'Tablet-Standard-64GB'
 'Blender-XL-Red' 'Vacuum-Pro-Black' 'Sneakers-Urban-42' 'Jacket-Winter-M']


In [24]:
# Eliminar las 101 filas donde hay nulos en el dataset de marketing
marketing = marketing.dropna()

# Comprobación de que el conteo de nulos haya bajado a cero
print("Nulos actuales en marketing:")
print(marketing.isnull().sum())

Nulos actuales en marketing:
fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en PowerBI y crear un dashsboard.

In [25]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

Se calculan los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [26]:
# Calcular el ingreso total (Revenue)
ingreso_total = orders['monto_total'].sum()
print(f"Ingreso Total (Revenue): ${ingreso_total:,.2f}")

Ingreso Total (Revenue): $9,610,018.94


In [27]:
# Calcular la inversión total en marketing
inversion_marketing = marketing['gasto'].sum()

print(f"Inversión Total en Marketing: ${inversion_marketing:,.2f}")

Inversión Total en Marketing: $2,694,664.43


In [28]:
# A) Unión de las órdenes con el catálogo para traer el 'costo_unitario' de cada producto
orders_with_cost = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')

# B) Calcular el costo de los productos vendidos para cada fila (cantidad vendida x costo de adquisición)
orders_with_cost['costo_total_productos'] = orders_with_cost['cantidad'] * orders_with_cost['costo_unitario']

# C) Suma de todos los costos de productos
costo_productos_total = orders_with_cost['costo_total_productos'].sum()

# D) Calcular el costo totalcombinado
costo_total_negocio = costo_productos_total + inversion_marketing

# E) Profit (Ganancia Neta = Ingresos - Costos Totales)
profit = ingreso_total - costo_total_negocio

print(f"Costo Total de Productos : ${costo_productos_total:,.2f}")
print(f"Costo Total Combinado (Productos + Marketing): ${costo_total_negocio:,.2f}")
print(f"---")
print(f"Rentabilidad Neta (Profit): ${profit:,.2f}")

if profit > 0:
    print("El negocio SÍ es rentable")
else:
    print("El negocio NO es rentable")

Costo Total de Productos : $3,828,869.01
Costo Total Combinado (Productos + Marketing): $6,523,533.44
---
Rentabilidad Neta (Profit): $3,086,485.50
El negocio SÍ es rentable


In [29]:
# Calcular el ticket promedio
ticket_promedio = orders['monto_total'].mean()
print(f"Ticket Promedio por Orden: ${ticket_promedio:,.2f}")

Ticket Promedio por Orden: $385.85


In [30]:
# Calcular la cantidad promedio de artículos por orden
cantidad_promedio = orders['cantidad'].mean()

print(f"Cantidad Promedio de Productos por Orden: {cantidad_promedio:,.2f} piezas")

Cantidad Promedio de Productos por Orden: 1.50 piezas


In [31]:
# Producto más vendido:
# Agrupamos productos, sumamos las cantidades y ordenamos de mayor a menor
productos_vendidos = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

# Tomamos el nombre del producto más vendido y su cantidad total
producto_top_nombre = productos_vendidos.index[0]
producto_top_cantidad = productos_vendidos.iloc[0]

print(f"Producto más vendido: {producto_top_nombre} ({producto_top_cantidad:,.0f} unidades vendidas)")

Producto más vendido: Vacuum-Pro-Black (6,284 unidades vendidas)


In [32]:
# Gasto de Marketing por canal:
# Agrupamos por canal y sumamos la inversión, ordenando de mayor a menor
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print("--- Gasto en Marketing por Canal ---")
# Iteramos sobre el resultado para mostrarlo con un formato limpio
for canal, gasto in gasto_por_canal.items():
    print(f"• {canal.title()}: ${gasto:,.2f}")

--- Gasto en Marketing por Canal ---
• Social: $918,043.21
• Organic: $913,533.01
• Paid_Search: $863,088.21


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [33]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [34]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [35]:
# Explorar tabla events
# ======================
query_eventos_unicos = '''
SELECT DISTINCT nombre_evento
FROM events;
'''

events_unicos = pd.read_sql(query_eventos_unicos, con=engine)
events_unicos

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [36]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario)
FROM events
WHERE nombre_evento IN ('first_visit','select_item','add_to_cart','begin_checkout','add_payment_info','purchase')
GROUP BY nombre_evento
ORDER BY count DESC
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,count
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [37]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'first_visit'
),
cte_add_to_cart AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_to_cart'
),
cte_select_item AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'select_item'
),
cte_begin_checkout AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'begin_checkout'
),
cte_add_payment_info AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
  SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'purchase'
)
SELECT
  -- 1. Conteos de usuarios únicos ordenados por comportamiento de los datos
  (SELECT COUNT(*) FROM cte_first_visit) AS first_visit_users,
  (SELECT COUNT(*) FROM cte_add_to_cart) AS cart_users,
  (SELECT COUNT(*) FROM cte_select_item) AS select_item_users,
  (SELECT COUNT(*) FROM cte_begin_checkout) AS checkout_users,
  (SELECT COUNT(*) FROM cte_add_payment_info) AS payment_users,
  (SELECT COUNT(*) FROM cte_purchase) AS purchase_users,

  -- 2. Cálculos de Abandono respecto al paso anterior
  -- De First Visit a Add to Cart
  ((SELECT COUNT(*) FROM cte_first_visit) - (SELECT COUNT(*) FROM cte_add_to_cart)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_first_visit), 0) AS dropoff_after_visit_pct,

  -- De Add to Cart a Select Item
  ((SELECT COUNT(*) FROM cte_add_to_cart) - (SELECT COUNT(*) FROM cte_select_item)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_add_to_cart), 0) AS dropoff_after_cart_pct,

  -- De Select Item a Begin Checkout
  ((SELECT COUNT(*) FROM cte_select_item) - (SELECT COUNT(*) FROM cte_begin_checkout)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_select_item), 0) AS dropoff_after_select_pct,

  -- De Begin Checkout a Add Payment Info
  ((SELECT COUNT(*) FROM cte_begin_checkout) - (SELECT COUNT(*) FROM cte_add_payment_info)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_begin_checkout), 0) AS dropoff_after_checkout_pct,

  -- De Add Payment Info a Purchase
  ((SELECT COUNT(*) FROM cte_add_payment_info) - (SELECT COUNT(*) FROM cte_purchase)) * 100.0
    / NULLIF((SELECT COUNT(*) FROM cte_add_payment_info), 0) AS dropoff_after_payment_pct;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit_users,cart_users,select_item_users,checkout_users,payment_users,purchase_users,dropoff_after_visit_pct,dropoff_after_cart_pct,dropoff_after_select_pct,dropoff_after_checkout_pct,dropoff_after_payment_pct
0,7796,7634,7582,7208,6250,6240,2.077989,0.681163,4.932735,13.290788,0.16


---

# 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

Se busca analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [38]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [39]:
query_users_valida = '''
SELECT
    id_usuario,
    CAST(fecha_registro AS DATE) AS fecha_registro,
    país,
    dispositivo,
    tipo_plan
FROM users;
'''

users = pd.read_sql(query_users_valida, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [40]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head()

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [41]:
query_valida_fecha = '''
SELECT *
FROM user_activity
WHERE CAST(fecha_actividad AS DATE) >= '2025-01-01';
'''
resultado_actividad = pd.read_sql(query_valida_fecha, con=engine)
resultado_actividad.head()

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [42]:
# Unir las tablas users y user_activity para el análisis de cohortes
# ====================================================================
query_cohortes_join = '''
SELECT
    u.id_usuario,
    CAST(u.fecha_registro AS DATE) AS fecha_registro,
    u.tipo_plan,
    CAST(ua.fecha_actividad AS DATE) AS fecha_actividad,
    ua.dias_despues_registro,
    ua.activo
FROM users AS u
JOIN user_activity AS ua
    ON u.id_usuario = ua.id_usuario;
'''
cohortes_base = pd.read_sql(query_cohortes_join, con=engine)
cohortes_base.head()

,id_usuario,fecha_registro,tipo_plan,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-01-29,free,2025-02-05,7,0
1,user_0,2025-01-29,free,2025-02-12,14,1
2,user_0,2025-01-29,free,2025-02-19,21,1
3,user_0,2025-01-29,free,2025-02-26,28,0
4,user_1,2025-01-07,free,2025-01-14,7,0


In [43]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        u.id_usuario,
        DATE_TRUNC('week', CAST(u.fecha_registro AS DATE)) AS cohorte_semana,
        ua.dias_despues_registro,
        ua.activo
    FROM users AS u
    JOIN user_activity AS ua
        ON u.id_usuario = ua.id_usuario
),
conteos_base AS (
    SELECT
        cohorte_semana,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN id_usuario END) AS retenido_w3
    FROM cohortes
    GROUP BY cohorte_semana
)
SELECT
    cohorte_semana,
    usuarios_iniciales,

    -- Calculamos los porcentajes
    ROUND((retenido_w1 * 100.0) / NULLIF(usuarios_iniciales, 0), 2) AS semana_1,
    ROUND((retenido_w2 * 100.0) / NULLIF(usuarios_iniciales, 0), 2) AS semana_2,
    ROUND((retenido_w3 * 100.0) / NULLIF(usuarios_iniciales, 0), 2) AS semana_3

FROM conteos_base
ORDER BY cohorte_semana;
'''
# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,cohorte_semana,usuarios_iniciales,semana_1,semana_2,semana_3
0,2024-12-30 00:00:00+00:00,236,41.95,38.56,40.25
1,2025-01-06 00:00:00+00:00,351,41.31,44.73,42.17
2,2025-01-13 00:00:00+00:00,362,43.65,38.12,41.99
3,2025-01-20 00:00:00+00:00,394,44.16,39.59,39.85
4,2025-01-27 00:00:00+00:00,373,41.55,47.45,39.95
5,2025-02-03 00:00:00+00:00,405,40.49,43.70,45.43
6,2025-02-10 00:00:00+00:00,364,42.03,39.84,45.05
7,2025-02-17 00:00:00+00:00,338,47.04,40.24,39.94
8,2025-02-24 00:00:00+00:00,353,42.21,41.93,42.78
9,2025-03-03 00:00:00+00:00,363,39.67,43.53,40.22


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** Las compras se mantienen iguales en ambos casos
   - **H₁ (Hipótesis alternativa):** Las compras aumentaron con el cambio en el producto
   
**Test estadístico:** Prueba z  
**Nivel de significancia alpha:** 0.05

In [44]:
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [45]:
conversiones = experiment.groupby('variante')['convirtio'].sum()
conversiones

,convirtio
variante,
control,779
tratamiento,820


In [46]:
totales = experiment.groupby('variante')['convirtio'].count()
totales

,convirtio
variante,
control,4965
tratamiento,5035


In [47]:
exitos = [conversiones['control']], [conversiones['tratamiento']]
exitos

([np.int64(779)], [np.int64(820)])

In [48]:
observaciones = [totales['control']], [totales['tratamiento']]
observaciones

([np.int64(4965)], [np.int64(5035)])

In [49]:
z_stat, p_value = proportions_ztest(exitos, observaciones)

print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

Estadístico z: [-0.8132783]
Valor p: [0.41605852]


In [50]:
alpha = 0.05

if p_value < alpha:
    print("Rechazamos hipotesis nula: hay evidencia de una diferencia")
else:
    print("No rechazamos la hipotesis nula: no hay evidencia suficiente de una diferencia")

No rechazamos la hipotesis nula: no hay evidencia suficiente de una diferencia


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)


Se usan los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

**Resumen ejecutivo**

1.  Los datos presentaban algunas irregularidades que no los hacian confiables, por lo que se realizo la limpieza necesaria.
Los formatos de fechas estaban mal asi que se corrigieron, se eliminaron algunos outliers extremos en la columna de cantidades y algunos numeros negativos. Además se eliminaron también algunos valores nulos en cantidad, nombre producto y categoria.Los demas nulos se dejaron ya que no afectan al resto del analisis.
2.    Analizando los ingresos y costos de la empresa los datos indican que SI estamos ganando dinero con un profit de 3mill.  Revisando el funnel de conversión, el punto donde tenemos mayor porcentaje de abandono es pasando del checkout al pago con un 13.29% de abandono. Valdría la pena reviar si el proceso de pago es muy complejo o existe alguna falla para mejorar la efectividad de conversión.
3.   Revisando el funnel de conversión, el punto donde tenemos mayor porcentaje de abandono es pasando del checkout al pago con un 13.29% de abandono. Valdría la pena reviar si el proceso de pago es muy complejo o existe alguna falla para mejorar la efectividad de conversión.
4.   En cuanto al analisis de cohortes vale la pena señalar que la retención es siempre mayor a un 35%, llegando hasta un 45% y a lo largo del tiempo todas se estabilizan, lo que indica que los usuarios si regresan y se vuelven clientes frecuentes.
5.  Por otra parte los resultados del experimento A\B  sugiere que las diferencias en el numero de conversiones pueden deberse mas al azar que a los cambios implementados.
6.   Revisando el revenue de los datos, se observa una caida en el mes de febrero siendo el mas bajo de todos, pero marzo en marzo se alcanzo el valor mas alto y a partir de ahi se mantuvo estable, como sugerecnia hay que revisar si algun error en la plataforma genero esa caida para que no se repita.
7. Por último, todas las categorías generan un profit similar y también tienen cantidades vendidas similares, pero hay un producto que genero perdidas ya que su costo es mayor al revenue que deja para la empresa (laptop-gaming-16gb), Se debe revisar y corregir el precio para que deje de generar perdidas, además de implementar un sistema de control para que esto no se repita con algun otro producto.










### 📎 Enlace del Dashboard

In [53]:
# https://drive.google.com/drive/folders/1FO2C5cQUyz-vVwjL7OWycvhPVxJs14Ul?usp=sharing
# link de power bi